# 03 Constrained Decoding & LangChain Structured Output Enforcement

Implementing LangChain `PydanticOutputParser`, `.with_structured_output()`, FSA state trackers, and PyTorch `SchemaConstrainedLogitProcessor` overlays.

## Setup: Repository-Level Dotenv Configuration

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from repo-level .env file
load_dotenv()

print("Environment Setup Completed.")
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))

## Technique 1: LangChain PydanticOutputParser Schema Injection

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field

class UserExtraction(BaseModel):
    user_name: str = Field(description="Full legal name of person")
    age: int = Field(description="Age in years")
    is_active: bool = Field(description="Whether user status is active")

parser = PydanticOutputParser(pydantic_object=UserExtraction)

prompt_template = PromptTemplate(
    template="Extract user entity from text.\n{format_instructions}\nText: {query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

formatted_prompt = prompt_template.format(query="Alice Smith is 29 years old and currently active.")
print("LangChain Pydantic Prompt Template:\n", formatted_prompt)

## Technique 2: LangChain .with_structured_output() Model Binding

In [ ]:
import os

print("=== Calling LLM with Structured Output Parsing ===")
try:
    from langchain_openai import ChatOpenAI
    openai_key = os.getenv("OPENAI_API_KEY")
    if openai_key:
        llm = ChatOpenAI(model="gpt-4o-mini", api_key=openai_key)
        structured_llm = llm.with_structured_output(UserExtraction)
        res = structured_llm.invoke("Alice Smith is 29 years old and currently active.")
        print("Parsed Pydantic Output Object:", res)
    else:
        print("OPENAI_API_KEY not found in .env file.")
except Exception as e:
    print("LangChain structured output error:", e)

## Technique 3: PyTorch SchemaConstrainedLogitProcessor Overlay

In [ ]:
import torch
import torch.nn.functional as F

class SchemaConstrainedLogitProcessor:
    def __init__(self, allowed_ids: list[int]):
        self.allowed_ids = set(allowed_ids)

    def __call__(self, logits: torch.Tensor) -> torch.Tensor:
        mask = torch.full_like(logits, float('-inf'))
        for token_id in self.allowed_ids:
            mask[..., token_id] = 0.0
        return logits + mask

raw_logits = torch.tensor([4.2, 3.8, 5.1, 4.9, 3.2, 2.1])
allowed_tokens = [0, 1, 5]

processor = SchemaConstrainedLogitProcessor(allowed_ids=allowed_tokens)
constrained_logits = processor(raw_logits)

raw_probs = F.softmax(raw_logits, dim=-1).numpy()
constrained_probs = F.softmax(constrained_logits, dim=-1).numpy()

print(f"Raw Softmax Probs:         {raw_probs.round(3)}")
print(f"Constrained Softmax Probs: {constrained_probs.round(3)}")

## Part 4: Visualizing Logit Bias Sampling Probabilities

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.style.use('dark_background')
token_labels = ['"name"', '"age"', 'class', 'SELECT', 'WHERE', '{']
x = np.arange(len(token_labels))
width = 0.35

plt.figure(figsize=(9, 4))
plt.bar(x - width/2, raw_probs, width, label='Unconstrained Softmax', color='#ff6b6b', alpha=0.85)
plt.bar(x + width/2, constrained_probs, width, label='Constrained Logit Bias (-inf)', color='#51cf66', alpha=0.85)

plt.xticks(x, token_labels, fontsize=10, fontweight='bold')
plt.title('Logit Bias Masking: Enforcing 100% Valid Syntax Tokens')
plt.ylabel('Sampling Probability P(token)')
plt.legend(); plt.grid(True, axis='y'); plt.tight_layout(); plt.show()